In [1]:
# === SETUP: Run this first! ===
import os
import sys

# Change to project root and add to Python path
notebook_dir = os.getcwd()
project_root = os.path.dirname(notebook_dir)  # Goes up one level from 'notebooks/'
os.chdir(project_root)
sys.path.insert(0, project_root)

print(f"Project root: {project_root}")
print(f"tsnn module path: {os.path.join(project_root, 'tsnn')}")

Project root: /Users/gremy/Code/TSNN-1
tsnn module path: /Users/gremy/Code/TSNN-1/tsnn


In [2]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.linear_model import LassoCV, RidgeCV, LinearRegression
import torch
from torch.utils.data import Dataset, random_split
from torch.utils.data import DataLoader
import importlib
import sys
sys.path.append('/Users/cyrilgarcia/notebooks/tsnn/')

import tsnn

from tsnn.generators import generators
from tsnn import tstorch
from tsnn.benchmarks import benchmark_comparison, ml_benchmarks, torch_benchmarks
from tsnn import utils
from tsnn.tstorch import transformers
import torch.nn.functional as F
import math
from typing import Optional

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.neural_network import MLPRegressor

from torch import nn
from tqdm import tqdm
device = 'mps'


plt.style.use('ggplot')

In [3]:
# This is the new version of the notebook to make all the figures for the paper.

In [4]:
from tsnn.tstorch import models
from tsnn.benchmarks import torch_benchmarks
from tsnn.benchmarks.torch_benchmarks import MSELossWithL1Sparsity

In [5]:
# Notebook to run all the experiments for the paper.

# Setup

For now we will be working with the following effects:
- Simple linear dependency
- TS shift with lags chosen randomly for each feature, but constant accros stocks
- CS shift, with lags chosen randomly for each stock and each feature
- TS-CS shift: combination of the two lags above
- Conditioning of one feature by another

We will fix the following global correlation levels: $\rho$ = 1,2,5,10,20,50. The first and last might not be too meaningful.

Next let's fix the size of the data that we want to test. We will always work with $T=4000$ points, and use $2500$ for training, $1500$ for testing. For the other dimensions:
- Default 1: n_ts = 10, n_rolling = 10, n_f = 20. This gives $\gamma = 0.8$.
- Default 2: n_ts = 10, n_rolling = 10, n_f = 5. This gives $\gamma = 0.2$.

Could also add two extra cases to test larger n_rolling or n_ts:
- Long TS: n_ts = 10, n_rolling = 40, n_f = 5. This gives $\gamma = 0.8$.
- Long CS: n_ts = 40, n_rolling = 10, n_f = 5. This gives $\gamma = 0.8$.

Can also run an experiment varying jointly $\rho$ and $\gamma$ at a fixed theo correl level.

Next we need to compute a $\rho$ per feature. In the case where n_f=5, we will simply take $\frac{\rho}{\sqrt{5}}$ per feature. In the case n_f=20, we will take $\frac{\rho}{\sqrt{10}}$ for half of the features and $0$ for the half.

In [6]:
# Some basic functions.

In [7]:
def causal_mask(b, h, q_idx, kv_idx):
    return q_idx >= kv_idx

def causal_mask_radius(b, h, q_idx, kv_idx):
    return (q_idx >= kv_idx) & (q_idx <= kv_idx+1)

def plot_mask(mask_fn, seq_len=20, title=None, device="cpu"):
    """
    Plot a binary attention mask defined by mask_fn(b,h,q_idx,kv_idx)
    as a (seq_len x seq_len) matrix.
    """
    q_idx = torch.arange(seq_len, device=device)
    kv_idx = torch.arange(seq_len, device=device)
    b = torch.zeros(1, device=device) 
    h = torch.zeros(1, device=device)

    mask = mask_fn(b, h, q_idx[:, None], kv_idx[None, :])
    mask = mask.float().cpu()

    return pd.DataFrame(mask).style.background_gradient(axis=None).format(precision=0)

def build_attention_mask(mask_fn, seq_len, device="cpu"):
    q_idx = torch.arange(seq_len, device=device)
    kv_idx = torch.arange(seq_len, device=device)
    b = torch.zeros(1, device=device)
    h = torch.zeros(1, device=device)
    mask_bool = mask_fn(b, h, q_idx[:, None], kv_idx[None, :])  # (seq_len, seq_len)
    return mask_bool

In [8]:
def get_ols_corr(N_fea, T_train, rho, oos=True):
    ratio = N_fea/T_train
    if oos:
        return rho / np.sqrt(rho**2 + (1-rho**2) * ratio/(1-ratio))
    else:
        return rho / np.sqrt(rho**2 + (1-rho**2) * ratio)


In [9]:
def keep_topk_per_row3(x, k=3):
    vals, idx = torch.topk(x, k=k, dim=-1, largest=True)
    out = torch.zeros_like(x)
    out.scatter_(-1, idx, 1)
    return out

In [10]:
def keep_by_max_value2(x, frac=0.2):
    row_max = x.max(dim=-1, keepdim=True).values
    threshold = frac * row_max
    out = (x >= threshold).to(x.dtype)
    return out


# Run of all models on all effects

In [11]:
def run_models1(rhos, effect, T=4000, n_ts=10, n_f=5, n_rolling=10):

    list_effects1 = [effect]*n_f

    if n_f >=10:
        half_n_f = int(n_f/2)
        correl_split_by_fea1 = [1/np.sqrt(half_n_f)]*(half_n_f) + [0]*half_n_f 
    else:
        correl_split_by_fea1 = [1/np.sqrt(n_f)]*n_f

    mask = causal_mask
    mask_c = build_attention_mask(mask, n_rolling, device=device)
    
    z = generators.Generator(T, n_ts, n_f)
    
    res_train = []
    res_test = []

    for (i,rho) in enumerate(rhos):
        print("Running correl level:", rho)
        theo_correl_is = get_ols_corr(n_ts*n_f*n_rolling, T*0.625, rho, oos=False)
        theo_correl_oos = get_ols_corr(n_ts*n_f*n_rolling, T*0.625, rho, oos=True)


        z.generate_dataset_gr_simple(
            global_corr=rho, 
            correl_split_by_fea=correl_split_by_fea1, 
            list_type_effects=list_effects1,
            list_type_interaction=["cond"], 
            random_ts_shift=n_rolling,
        )


        models_torch = {
        'Global_MLP': models.GlobalMLP(n_ts, n_f, n_rolling, dropout=0.1).to(device),
        '1D_Trans_TT': models.OneDimensionalTransformer(n_ts, n_f, n_rolling, mask=mask_c, attn_direction="T",  num_attn_layers=2,
                                        d_model=64, dim_feedforward=256, nhead=8, compression="MLP", num_mlp_layers=2,
                                        dropout=0.1, roll_y=True).to(device),
        '1D_Trans_CC': models.OneDimensionalTransformer(n_ts, n_f, n_rolling, mask=mask_c, attn_direction="C",  num_attn_layers=2,
                                        d_model=64, dim_feedforward=256, nhead=8, compression="MLP", num_mlp_layers=2,
                                        dropout=0.1,  roll_y=False).to(device),                                
        '2D_Trans_TC': models.CustomBiDimensionalTransformer(
            n_ts, n_f, n_rolling, mask=mask_c, layers='TC', nhead=8,
            dropout=0.1, d_model=64, dim_feedforward=256, sparsify=None, roll_y=True, embeddings="both",
        ).to(device),
        '2D_Trans_TCTC': models.CustomBiDimensionalTransformer(
            n_ts, n_f, n_rolling, mask=mask_c, layers='TCTC', nhead=8,
            dropout=0.1, d_model=64, dim_feedforward=256, sparsify=None, roll_y=True, embeddings="both",
        ).to(device),
        '1D_Trans_TT_sparse': models.OneDimensionalTransformer(n_ts, n_f, n_rolling, mask=mask_c, attn_direction="T",  num_attn_layers=2,
                                        d_model=64, dim_feedforward=256, nhead=8, compression="MLP", num_mlp_layers=2,
                                        dropout=0.1, roll_y=True, sparsify=keep_topk_per_row3,).to(device),
        '1D_Trans_CC_sparse': models.OneDimensionalTransformer(n_ts, n_f, n_rolling, mask=mask_c, attn_direction="C",  num_attn_layers=2,
                                        d_model=64, dim_feedforward=256, nhead=8, compression="MLP", num_mlp_layers=2,
                                        dropout=0.1,  roll_y=False, sparsify=keep_topk_per_row3,).to(device),                                
        '2D_Trans_TC_sparse': models.CustomBiDimensionalTransformer(
            n_ts, n_f, n_rolling, mask=mask_c, layers='TC', nhead=8,
            dropout=0.1, d_model=64, dim_feedforward=256, sparsify=keep_topk_per_row3, roll_y=True, embeddings="both",
        ).to(device),
        '2D_Trans_TCTC_sparse': models.CustomBiDimensionalTransformer(
            n_ts, n_f, n_rolling, mask=mask_c, layers='TCTC', nhead=8,
            dropout=0.1, d_model=64, dim_feedforward=256, sparsify=keep_topk_per_row3, roll_y=True, embeddings="both",
        ).to(device),
        }

        models_torch = {
            k: torch_benchmarks.TorchWrapper(
                models_torch[k], 
                optimizer=torch.optim.AdamW(models_torch[k].parameters(), lr=0.001),
                loss_fn=nn.MSELoss()  
            ) for k in models_torch
        }

        
        for k in models_torch:
            roll_y = True
            if k in ['Global_MLP', '1D_Trans_CC', '1D_Trans_CC_sparse']:
                roll_y = False
            z.get_dataloader(n_rolling=n_rolling, roll_y=roll_y)
            epochs=40
            if k in ['1D_Trans_CC', '1D_Trans_CC_sparse']:
                epochs=80
            models_torch[k].fit(z.train, test=z.test, epochs=epochs, plot=False, verbose=0)
            
      
        comp = benchmark_comparison.Comparator(models=[models_torch[k] for k in models_torch], 
        model_names=[k for k in models_torch]
        )

        corr_train = comp.correl(z, mode="train", return_values=True)
        corr_test  = comp.correl(z, mode="test",  return_values=True)

        for k in models_torch:

            train_corr = corr_train.loc[k, "optimal"]
            test_corr  = corr_test.loc[k, "optimal"]

            res_train.append({
                "rho": rhos[i],
                "model": k,
                "train_corr_optimal": train_corr
            })
            res_test.append({
                "rho": rhos[i],
                "model": k,
                "test_corr_optimal": test_corr
            })


        res_train.append({
                "rho": rhos[i],
                "model": "theo_correl",
                "train_corr_optimal": theo_correl_is
            })
        res_test.append({
                "rho": rhos[i],
                "model": "theo_correl",
                "test_corr_optimal": theo_correl_oos
            })

    
    res_train = pd.DataFrame(res_train)
    res_test  = pd.DataFrame(res_test)

    res_train = res_train.pivot(index="rho", columns="model", values="train_corr_optimal")
    res_test  = res_test.pivot(index="rho", columns="model", values="test_corr_optimal")

    return res_train, res_test

## n_f = 5

In [ ]:
#all_effects_nf5_train_dic = {}
#all_effects_nf5_test_dic = {}

In [13]:
for effect in ['lin', 'TS_shift', 'CS_shift', 'fea_cond', 'TSCS_shift', 'TS_cond', 'CS_cond']:
    print("Running_effect:", effect)
    out_train1, out_test1 = run_models1([0.02, 0.05, 0.1, 0.2, 0.5], effect, T=4000, n_ts=10, n_f=5, n_rolling=10)
    all_effects_nf5_train_dic[effect] = out_train1
    all_effects_nf5_test_dic[effect] = out_test1

Running_effect: lin
Running correl level: 0.02


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.05


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.1


...

/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.2


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.5


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


In [14]:
for effect in ['lin', 'TS_shift', 'CS_shift', 'fea_cond', 'TSCS_shift', 'TS_cond', 'CS_cond']:
    print(effect)
    display(all_effects_nf5_train_dic[effect].T.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))
    display(all_effects_nf5_test_dic[effect].T.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

lin


rho,0.020000,0.050000,0.100000,0.200000,0.500000
model,,,,,
1D_Trans_CC,0.037,0.091,0.142,0.278,0.640
1D_Trans_CC_sparse,0.039,0.070,0.174,0.283,0.687
1D_Trans_TT,0.043,0.091,0.180,0.324,0.713
1D_Trans_TT_sparse,0.043,0.094,0.184,0.337,0.711
2D_Trans_TC,0.091,0.171,0.413,0.576,0.916
2D_Trans_TCTC,0.098,0.216,0.355,0.452,0.883
2D_Trans_TCTC_sparse,0.091,0.230,0.263,0.517,0.907
2D_Trans_TC_sparse,0.084,0.228,0.344,0.660,0.958
Global_MLP,0.030,0.052,0.110,0.201,0.504


rho,0.020000,0.050000,0.100000,0.200000,0.500000
model,,,,,
1D_Trans_CC,0.048,0.117,0.176,0.329,0.699
1D_Trans_CC_sparse,0.043,0.068,0.208,0.344,0.732
1D_Trans_TT,0.041,0.097,0.196,0.353,0.728
1D_Trans_TT_sparse,0.050,0.104,0.193,0.364,0.728
2D_Trans_TC,0.090,0.161,0.411,0.578,0.917
2D_Trans_TCTC,0.101,0.214,0.370,0.439,0.886
2D_Trans_TCTC_sparse,0.072,0.227,0.269,0.527,0.908
2D_Trans_TC_sparse,0.085,0.224,0.334,0.664,0.959
Global_MLP,0.049,0.086,0.163,0.312,0.651


TS_shift


rho,0.020000,0.050000,0.100000,0.200000,0.500000
model,,,,,
1D_Trans_CC,0.040,0.078,0.151,0.295,0.661
1D_Trans_CC_sparse,0.041,0.075,0.154,0.298,0.679
1D_Trans_TT,0.015,0.042,0.078,0.178,0.444
1D_Trans_TT_sparse,0.012,0.042,0.081,0.162,0.433
2D_Trans_TC,0.024,0.118,0.054,0.442,0.833
2D_Trans_TCTC,0.022,0.091,0.075,0.392,0.828
2D_Trans_TCTC_sparse,0.022,0.128,0.171,0.450,0.841
2D_Trans_TC_sparse,0.033,0.174,0.245,0.545,0.872
Global_MLP,0.024,0.054,0.103,0.208,0.509


...

rho,0.020000,0.050000,0.100000,0.200000,0.500000
model,,,,,
1D_Trans_CC,0.028,0.037,0.095,0.185,0.559
1D_Trans_CC_sparse,0.016,0.039,0.095,0.189,0.572
1D_Trans_TT,0.011,0.026,0.076,0.147,0.386
1D_Trans_TT_sparse,0.014,0.032,0.074,0.141,0.378
2D_Trans_TC,0.010,0.013,0.031,0.080,0.411
2D_Trans_TCTC,0.010,0.015,-0.003,0.066,0.390
2D_Trans_TCTC_sparse,-0.010,0.023,0.037,0.083,0.398
2D_Trans_TC_sparse,-0.001,0.011,0.019,0.073,0.339
Global_MLP,0.021,0.041,0.102,0.190,0.504


rho,0.020000,0.050000,0.100000,0.200000,0.500000
model,,,,,
1D_Trans_CC,0.018,0.016,0.028,0.073,0.421
1D_Trans_CC_sparse,0.028,0.022,0.035,0.082,0.437
1D_Trans_TT,0.005,0.009,0.006,0.002,0.006
1D_Trans_TT_sparse,-0.002,-0.007,0.006,-0.003,0.003
2D_Trans_TC,0.012,0.003,0.031,-0.001,0.307
2D_Trans_TCTC,-0.012,-0.000,0.011,0.004,0.273
2D_Trans_TCTC_sparse,0.005,0.015,0.043,0.031,0.248
2D_Trans_TC_sparse,-0.006,-0.004,0.023,0.040,0.267
Global_MLP,-0.001,0.008,0.012,0.010,0.018


CS_cond


rho,0.020000,0.050000,0.100000,0.200000,0.500000
model,,,,,
1D_Trans_CC,0.016,0.046,0.084,0.178,0.425
1D_Trans_CC_sparse,0.012,0.047,0.088,0.175,0.407
1D_Trans_TT,0.017,0.052,0.086,0.179,0.447
1D_Trans_TT_sparse,0.017,0.050,0.082,0.185,0.446
2D_Trans_TC,0.010,0.017,0.066,0.139,0.438
2D_Trans_TCTC,0.022,0.034,0.078,0.064,0.407
2D_Trans_TCTC_sparse,0.005,0.042,0.063,0.153,0.496
2D_Trans_TC_sparse,0.010,0.034,0.041,0.095,0.447
Global_MLP,0.020,0.051,0.100,0.206,0.500


rho,0.020000,0.050000,0.100000,0.200000,0.500000
model,,,,,
1D_Trans_CC,-0.008,-0.009,-0.008,0.004,0.005
1D_Trans_CC_sparse,0.011,-0.009,0.013,-0.005,0.001
1D_Trans_TT,-0.005,-0.005,0.007,0.051,0.113
1D_Trans_TT_sparse,0.007,0.028,0.015,0.035,0.118
2D_Trans_TC,-0.011,0.002,0.016,0.072,0.349
2D_Trans_TCTC,0.026,0.025,0.041,0.028,0.248
2D_Trans_TCTC_sparse,0.002,0.030,0.026,0.078,0.380
2D_Trans_TC_sparse,-0.001,0.012,0.022,0.052,0.339
Global_MLP,-0.011,-0.010,-0.001,-0.004,0.010


### Hardcoded df

In [15]:
# Here let's give the python code to define a pandas table that has exactly the above parameters.

In [18]:
# Generate code to recreate the DataFrame with rounded values
def generate_dataframe_code(df, decimals=3, var_name='df'):
    """Generate Python code to recreate a DataFrame with rounded floats."""
    
    # Round the DataFrame
    df_rounded = df.round(decimals)
    
    # Convert to dictionary format
    data_dict = df_rounded.to_dict('list')
    
    # Format the code string
    code = f"{var_name} = pd.DataFrame({{\n"
    for col, values in data_dict.items():
        code += f"    '{col}': {values},\n"
    code += "})"
    
    return code


In [22]:
for effect in ['lin', 'TS_shift', 'CS_shift', 'fea_cond', 'TSCS_shift', 'TS_cond', 'CS_cond']:
    print(generate_dataframe_code(all_effects_nf5_test_dic[effect], var_name="df_test_" + effect))


df_test_lin = pd.DataFrame({
    '1D_Trans_CC': [0.048, 0.117, 0.176, 0.329, 0.699],
    '1D_Trans_CC_sparse': [0.043, 0.068, 0.208, 0.344, 0.732],
    '1D_Trans_TT': [0.041, 0.097, 0.196, 0.353, 0.728],
    '1D_Trans_TT_sparse': [0.05, 0.104, 0.193, 0.364, 0.728],
    '2D_Trans_TC': [0.09, 0.161, 0.411, 0.578, 0.917],
    '2D_Trans_TCTC': [0.101, 0.214, 0.37, 0.439, 0.886],
    '2D_Trans_TCTC_sparse': [0.072, 0.227, 0.269, 0.527, 0.908],
    '2D_Trans_TC_sparse': [0.085, 0.224, 0.334, 0.664, 0.959],
    'Global_MLP': [0.049, 0.086, 0.163, 0.312, 0.651],
    'theo_correl': [0.04, 0.1, 0.197, 0.378, 0.756],
})
df_test_TS_shift = pd.DataFrame({
    '1D_Trans_CC': [0.041, 0.091, 0.178, 0.342, 0.71],
    '1D_Trans_CC_sparse': [0.05, 0.067, 0.183, 0.352, 0.721],
    '1D_Trans_TT': [0.005, 0.018, -0.004, 0.055, 0.16],
    '1D_Trans_TT_sparse': [0.009, 0.004, -0.008, 0.046, 0.141],
    '2D_Trans_TC': [0.007, 0.092, 0.028, 0.437, 0.819],
    '2D_Trans_TCTC': [0.012, 0.096, 0.05, 0.368, 0.81],


In [21]:
# Train dataframes

df_train_lin = pd.DataFrame({
    '1D_Trans_CC': [0.037, 0.091, 0.142, 0.278, 0.64],
    '1D_Trans_CC_sparse': [0.039, 0.07, 0.174, 0.283, 0.687],
    '1D_Trans_TT': [0.043, 0.091, 0.18, 0.324, 0.713],
    '1D_Trans_TT_sparse': [0.043, 0.094, 0.184, 0.337, 0.711],
    '2D_Trans_TC': [0.091, 0.171, 0.413, 0.576, 0.916],
    '2D_Trans_TCTC': [0.098, 0.216, 0.355, 0.452, 0.883],
    '2D_Trans_TCTC_sparse': [0.091, 0.23, 0.263, 0.517, 0.907],
    '2D_Trans_TC_sparse': [0.084, 0.228, 0.344, 0.66, 0.958],
    'Global_MLP': [0.03, 0.052, 0.11, 0.201, 0.504],
    'theo_correl': [0.045, 0.111, 0.219, 0.415, 0.791],
})
df_train_TS_shift = pd.DataFrame({
    '1D_Trans_CC': [0.04, 0.078, 0.151, 0.295, 0.661],
    '1D_Trans_CC_sparse': [0.041, 0.075, 0.154, 0.298, 0.679],
    '1D_Trans_TT': [0.015, 0.042, 0.078, 0.178, 0.444],
    '1D_Trans_TT_sparse': [0.012, 0.042, 0.081, 0.162, 0.433],
    '2D_Trans_TC': [0.024, 0.118, 0.054, 0.442, 0.833],
    '2D_Trans_TCTC': [0.022, 0.091, 0.075, 0.392, 0.828],
    '2D_Trans_TCTC_sparse': [0.022, 0.128, 0.171, 0.45, 0.841],
    '2D_Trans_TC_sparse': [0.033, 0.174, 0.245, 0.545, 0.872],
    'Global_MLP': [0.024, 0.054, 0.103, 0.208, 0.509],
    'theo_correl': [0.045, 0.111, 0.219, 0.415, 0.791],
})
df_train_CS_shift = pd.DataFrame({
    '1D_Trans_CC': [0.029, 0.045, 0.102, 0.19, 0.605],
    '1D_Trans_CC_sparse': [0.038, 0.049, 0.108, 0.213, 0.63],
    '1D_Trans_TT': [0.053, 0.081, 0.169, 0.318, 0.718],
    '1D_Trans_TT_sparse': [0.049, 0.079, 0.173, 0.318, 0.728],
    '2D_Trans_TC': [0.063, 0.093, 0.292, 0.526, 0.867],
    '2D_Trans_TCTC': [0.008, 0.124, 0.29, 0.451, 0.804],
    '2D_Trans_TCTC_sparse': [0.094, 0.107, 0.234, 0.472, 0.817],
    '2D_Trans_TC_sparse': [0.06, 0.092, 0.238, 0.474, 0.855],
    'Global_MLP': [0.032, 0.046, 0.1, 0.194, 0.519],
    'theo_correl': [0.045, 0.111, 0.219, 0.415, 0.791],
})
df_train_fea_cond = pd.DataFrame({
    '1D_Trans_CC': [0.02, 0.045, 0.099, 0.259, 0.6],
    '1D_Trans_CC_sparse': [0.012, 0.05, 0.095, 0.258, 0.607],
    '1D_Trans_TT': [0.018, 0.042, 0.08, 0.201, 0.465],
    '1D_Trans_TT_sparse': [0.022, 0.039, 0.074, 0.203, 0.449],
    '2D_Trans_TC': [0.082, 0.17, 0.247, 0.558, 0.833],
    '2D_Trans_TCTC': [0.095, 0.117, 0.19, 0.411, 0.779],
    '2D_Trans_TCTC_sparse': [0.032, 0.149, 0.214, 0.489, 0.808],
    '2D_Trans_TC_sparse': [0.069, 0.198, 0.276, 0.571, 0.854],
    'Global_MLP': [0.017, 0.039, 0.094, 0.22, 0.504],
    'theo_correl': [0.045, 0.111, 0.219, 0.415, 0.791],
})
df_train_TSCS_shift = pd.DataFrame({
    '1D_Trans_CC': [0.02, 0.042, 0.1, 0.213, 0.584],
    '1D_Trans_CC_sparse': [0.013, 0.035, 0.103, 0.214, 0.626],
    '1D_Trans_TT': [0.019, 0.029, 0.072, 0.184, 0.424],
    '1D_Trans_TT_sparse': [0.015, 0.028, 0.072, 0.169, 0.404],
    '2D_Trans_TC': [0.005, 0.006, 0.026, 0.06, 0.172],
    '2D_Trans_TCTC': [-0.008, 0.013, 0.024, 0.032, 0.217],
    '2D_Trans_TCTC_sparse': [-0.001, 0.011, 0.032, 0.036, 0.176],
    '2D_Trans_TC_sparse': [-0.002, 0.018, 0.025, 0.041, 0.118],
    'Global_MLP': [0.021, 0.04, 0.1, 0.202, 0.508],
    'theo_correl': [0.045, 0.111, 0.219, 0.415, 0.791],
})
df_train_TS_cond = pd.DataFrame({
    '1D_Trans_CC': [0.028, 0.037, 0.095, 0.185, 0.559],
    '1D_Trans_CC_sparse': [0.016, 0.039, 0.095, 0.189, 0.572],
    '1D_Trans_TT': [0.011, 0.026, 0.076, 0.147, 0.386],
    '1D_Trans_TT_sparse': [0.014, 0.032, 0.074, 0.141, 0.378],
    '2D_Trans_TC': [0.01, 0.013, 0.031, 0.08, 0.411],
    '2D_Trans_TCTC': [0.01, 0.015, -0.003, 0.066, 0.39],
    '2D_Trans_TCTC_sparse': [-0.01, 0.023, 0.037, 0.083, 0.398],
    '2D_Trans_TC_sparse': [-0.001, 0.011, 0.019, 0.073, 0.339],
    'Global_MLP': [0.021, 0.041, 0.102, 0.19, 0.504],
    'theo_correl': [0.045, 0.111, 0.219, 0.415, 0.791],
})
df_train_CS_cond = pd.DataFrame({
    '1D_Trans_CC': [0.016, 0.046, 0.084, 0.178, 0.425],
    '1D_Trans_CC_sparse': [0.012, 0.047, 0.088, 0.175, 0.407],
    '1D_Trans_TT': [0.017, 0.052, 0.086, 0.179, 0.447],
    '1D_Trans_TT_sparse': [0.017, 0.05, 0.082, 0.185, 0.446],
    '2D_Trans_TC': [0.01, 0.017, 0.066, 0.139, 0.438],
    '2D_Trans_TCTC': [0.022, 0.034, 0.078, 0.064, 0.407],
    '2D_Trans_TCTC_sparse': [0.005, 0.042, 0.063, 0.153, 0.496],
    '2D_Trans_TC_sparse': [0.01, 0.034, 0.041, 0.095, 0.447],
    'Global_MLP': [0.02, 0.051, 0.1, 0.206, 0.5],
    'theo_correl': [0.045, 0.111, 0.219, 0.415, 0.791],
})

In [23]:
# Test data

df_test_lin = pd.DataFrame({
    '1D_Trans_CC': [0.048, 0.117, 0.176, 0.329, 0.699],
    '1D_Trans_CC_sparse': [0.043, 0.068, 0.208, 0.344, 0.732],
    '1D_Trans_TT': [0.041, 0.097, 0.196, 0.353, 0.728],
    '1D_Trans_TT_sparse': [0.05, 0.104, 0.193, 0.364, 0.728],
    '2D_Trans_TC': [0.09, 0.161, 0.411, 0.578, 0.917],
    '2D_Trans_TCTC': [0.101, 0.214, 0.37, 0.439, 0.886],
    '2D_Trans_TCTC_sparse': [0.072, 0.227, 0.269, 0.527, 0.908],
    '2D_Trans_TC_sparse': [0.085, 0.224, 0.334, 0.664, 0.959],
    'Global_MLP': [0.049, 0.086, 0.163, 0.312, 0.651],
    'theo_correl': [0.04, 0.1, 0.197, 0.378, 0.756],
})
df_test_TS_shift = pd.DataFrame({
    '1D_Trans_CC': [0.041, 0.091, 0.178, 0.342, 0.71],
    '1D_Trans_CC_sparse': [0.05, 0.067, 0.183, 0.352, 0.721],
    '1D_Trans_TT': [0.005, 0.018, -0.004, 0.055, 0.16],
    '1D_Trans_TT_sparse': [0.009, 0.004, -0.008, 0.046, 0.141],
    '2D_Trans_TC': [0.007, 0.092, 0.028, 0.437, 0.819],
    '2D_Trans_TCTC': [0.012, 0.096, 0.05, 0.368, 0.81],
    '2D_Trans_TCTC_sparse': [0.024, 0.103, 0.173, 0.453, 0.831],
    '2D_Trans_TC_sparse': [0.018, 0.173, 0.228, 0.545, 0.864],
    'Global_MLP': [0.044, 0.075, 0.157, 0.325, 0.67],
    'theo_correl': [0.04, 0.1, 0.197, 0.378, 0.756],
})
df_test_CS_shift = pd.DataFrame({
    '1D_Trans_CC': [0.016, 0.007, 0.04, 0.094, 0.576],
    '1D_Trans_CC_sparse': [0.021, 0.02, 0.043, 0.128, 0.604],
    '1D_Trans_TT': [0.059, 0.092, 0.193, 0.349, 0.738],
    '1D_Trans_TT_sparse': [0.065, 0.095, 0.168, 0.324, 0.738],
    '2D_Trans_TC': [0.055, 0.08, 0.263, 0.519, 0.86],
    '2D_Trans_TCTC': [0.002, 0.114, 0.277, 0.423, 0.798],
    '2D_Trans_TCTC_sparse': [0.096, 0.094, 0.225, 0.46, 0.816],
    '2D_Trans_TC_sparse': [0.056, 0.079, 0.216, 0.459, 0.847],
    'Global_MLP': [0.037, 0.068, 0.16, 0.298, 0.678],
    'theo_correl': [0.04, 0.1, 0.197, 0.378, 0.756],
})
df_test_fea_cond = pd.DataFrame({
    '1D_Trans_CC': [0.018, 0.022, 0.047, 0.189, 0.524],
    '1D_Trans_CC_sparse': [-0.002, 0.024, 0.048, 0.184, 0.52],
    '1D_Trans_TT': [-0.013, 0.028, 0.019, 0.047, 0.165],
    '1D_Trans_TT_sparse': [0.03, 0.009, 0.019, 0.058, 0.152],
    '2D_Trans_TC': [0.051, 0.159, 0.236, 0.543, 0.811],
    '2D_Trans_TCTC': [0.08, 0.111, 0.169, 0.415, 0.757],
    '2D_Trans_TCTC_sparse': [0.038, 0.145, 0.214, 0.484, 0.787],
    '2D_Trans_TC_sparse': [0.061, 0.207, 0.275, 0.572, 0.842],
    'Global_MLP': [0.001, 0.012, -0.0, -0.002, 0.019],
    'theo_correl': [0.04, 0.1, 0.197, 0.378, 0.756],
})
df_test_TSCS_shift = pd.DataFrame({
    '1D_Trans_CC': [0.024, 0.015, 0.048, 0.153, 0.546],
    '1D_Trans_CC_sparse': [0.003, 0.017, 0.06, 0.141, 0.6],
    '1D_Trans_TT': [0.004, 0.007, 0.006, 0.074, 0.068],
    '1D_Trans_TT_sparse': [0.01, -0.0, -0.001, 0.034, 0.056],
    '2D_Trans_TC': [0.003, -0.004, 0.006, 0.007, 0.012],
    '2D_Trans_TCTC': [0.012, 0.011, 0.007, -0.014, 0.02],
    '2D_Trans_TCTC_sparse': [-0.002, 0.001, -0.011, -0.005, 0.013],
    '2D_Trans_TC_sparse': [0.007, -0.008, 0.005, -0.012, 0.015],
    'Global_MLP': [0.041, 0.059, 0.149, 0.305, 0.671],
    'theo_correl': [0.04, 0.1, 0.197, 0.378, 0.756],
})
df_test_TS_cond = pd.DataFrame({
    '1D_Trans_CC': [0.018, 0.016, 0.028, 0.073, 0.421],
    '1D_Trans_CC_sparse': [0.028, 0.022, 0.035, 0.082, 0.437],
    '1D_Trans_TT': [0.005, 0.009, 0.006, 0.002, 0.006],
    '1D_Trans_TT_sparse': [-0.002, -0.007, 0.006, -0.003, 0.003],
    '2D_Trans_TC': [0.012, 0.003, 0.031, -0.001, 0.307],
    '2D_Trans_TCTC': [-0.012, -0.0, 0.011, 0.004, 0.273],
    '2D_Trans_TCTC_sparse': [0.005, 0.015, 0.043, 0.031, 0.248],
    '2D_Trans_TC_sparse': [-0.006, -0.004, 0.023, 0.04, 0.267],
    'Global_MLP': [-0.001, 0.008, 0.012, 0.01, 0.018],
    'theo_correl': [0.04, 0.1, 0.197, 0.378, 0.756],
})
df_test_CS_cond = pd.DataFrame({
    '1D_Trans_CC': [-0.008, -0.009, -0.008, 0.004, 0.005],
    '1D_Trans_CC_sparse': [0.011, -0.009, 0.013, -0.005, 0.001],
    '1D_Trans_TT': [-0.005, -0.005, 0.007, 0.051, 0.113],
    '1D_Trans_TT_sparse': [0.007, 0.028, 0.015, 0.035, 0.118],
    '2D_Trans_TC': [-0.011, 0.002, 0.016, 0.072, 0.349],
    '2D_Trans_TCTC': [0.026, 0.025, 0.041, 0.028, 0.248],
    '2D_Trans_TCTC_sparse': [0.002, 0.03, 0.026, 0.078, 0.38],
    '2D_Trans_TC_sparse': [-0.001, 0.012, 0.022, 0.052, 0.339],
    'Global_MLP': [-0.011, -0.01, -0.001, -0.004, 0.01],
    'theo_correl': [0.04, 0.1, 0.197, 0.378, 0.756],
})

### Creating figures for the paper

## n_f 20